In [1]:
import os
import json
import ast
import io
import time
from pathlib import Path
from collections import Counter
from contextlib import redirect_stdout
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import nglview as nv
from IPython.display import display, clear_output
from tqdm.auto import tqdm
from cathub.query import CathubQuery
from ase.io import read, write
from ase.visualize.plot import plot_atoms
from ase.data import covalent_radii
from dotenv import load_dotenv

In [2]:
load_dotenv()
DATA_DIR = Path("data/YohannesCombined2023")
STRUCTURE_DIR = DATA_DIR / "structures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
STRUCTURE_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
client = CathubQuery(api_key=os.getenv("CATHUB_API_KEY"))
metadata_path = DATA_DIR / "reactions.pkl"

if metadata_path.exists():
    df = pd.read_pickle(metadata_path)
    print("Loaded cached reactions")
else:
    print("Downloading all reaction metadata...")
    with redirect_stdout(io.StringIO()):
        df = client.get_reactions(pub_id="YohannesCombined2023", n_results="all", include_structures=False)

    df = df.reset_index(drop=True)
    df.to_pickle(metadata_path)
    print("Saved reaction metadata")

print(f"Reactions: {len(df):,}")
df.head()

Loaded cached reactions
Reactions: 12,935


,activation_energy,chemical_composition,coverages,dft_code,dft_functional,facet,products,pub_id,reactants,reaction_energy,sites,surface_composition,username,doi,atoms_name,atoms_id,equation
0,None,Mn16N16,"{""CHO"": 1}",VASP,PBE,100,"{""CHOstar"": 1}",YohannesCombined2023,"{""star"": 1, ""COgas"": 1, ""H2gas"": 0.5}",-2.483365,"{""CHO"": ""site4""}",MnN-mp-1009130-A,samira.siahrostami@ucalgary.ca,10.1021/acscatal.3c01249,"[star, H2gas, CHOstar, COgas]","[a6ecb9cd0dbc076ea3754f57d4baaebf, bbbcca193af...",CO(g) + 0.5H2(g) + * -> CHO*
1,None,Mn16N16,"{""CO"": 1}",VASP,PBE,100,"{""COstar"": 1}",YohannesCombined2023,"{""star"": 1, ""COgas"": 1}",-1.729397,"{""CO"": ""site4""}",MnN-mp-1009130-A,samira.siahrostami@ucalgary.ca,10.1021/acscatal.3c01249,"[star, COstar, COgas]","[a6ecb9cd0dbc076ea3754f57d4baaebf, c16318e1b77...",CO(g) + * -> CO*
2,None,Mn16N16,"{""CHO"": 1}",VASP,PBE,100,"{""CHOstar"": 1}",YohannesCombined2023,"{""star"": 1, ""COgas"": 1, ""H2gas"": 0.5}",-2.610425,"{""CHO"": ""site3""}",MnN-mp-1009130-A,samira.siahrostami@ucalgary.ca,10.1021/acscatal.3c01249,"[star, H2gas, CHOstar, COgas]","[a6ecb9cd0dbc076ea3754f57d4baaebf, bbbcca193af...",CO(g) + 0.5H2(g) + * -> CHO*
3,None,Mn16N16,"{""CHO"": 1}",VASP,PBE,100,"{""CHOstar"": 1}",YohannesCombined2023,"{""star"": 1, ""COgas"": 1, ""H2gas"": 0.5}",-2.700989,"{""CHO"": ""site1""}",MnN-mp-1009130-A,samira.siahrostami@ucalgary.ca,10.1021/acscatal.3c01249,"[CHOstar, star, H2gas, COgas]","[7b88650edc4f518f0381a3bd774acb86, a6ecb9cd0db...",CO(g) + 0.5H2(g) + * -> CHO*
4,None,Mn16N16,"{""CHO"": 1}",VASP,PBE,100,"{""CHOstar"": 1}",YohannesCombined2023,"{""star"": 1, ""COgas"": 1, ""H2gas"": 0.5}",3.189732,"{""CHO"": ""site6""}",MnN-mp-1009130-A,samira.siahrostami@ucalgary.ca,10.1021/acscatal.3c01249,"[CHOstar, star, H2gas, COgas]","[3589d73db7b2d73922b1ef7f934b0f03, a6ecb9cd0db...",CO(g) + 0.5H2(g) + * -> CHO*


In [5]:
df.columns

Index(['activation_energy', 'chemical_composition', 'coverages', 'dft_code',
       'dft_functional', 'facet', 'products', 'pub_id', 'reactants',
       'reaction_energy', 'sites', 'surface_composition', 'username', 'doi',
       'atoms_name', 'atoms_id', 'equation'],
      dtype='object')